In [ ]:
!pip install transformers==4.57.1
!pip install datasets tqdm bitsandbytes accelerate

In [ ]:
import os

os.environ["HF_HOME"] = "/home/ec2-user/SageMaker/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/home/ec2-user/SageMaker/.cache/huggingface"
os.environ["HF_TOKEN"] = ""

In [ ]:
!nvidia-smi

## Imports

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import BitsAndBytesConfig as TransformersBitsAndBytesConfig
from datasets import load_dataset

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "black-forest-labs/FLUX.2-klein-9B"

## Loading Quantised Text Encoder & Tokenizer

In [ ]:
qunatization_config = {
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_compute_dtype": torch.bfloat16,
}

In [ ]:
text_encoder = AutoModelForCausalLM.from_pretrained(
    model_id,
    subfolder="text_encoder",
    quantization_config=TransformersBitsAndBytesConfig(**qunatization_config),
    device_map="auto",
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, subfolder="tokenizer")

In [ ]:
total_params_text_encoder = sum([param.numel() for param in text_encoder.parameters()])
total_params_text_encoder

## Loading Raw Dataset

In [ ]:
dataset = load_dataset("derekl35/alphonse-mucha-style")

## Encoding Prompts with Text Encoder

In [ ]:
def encode_prompts(inp):
    with torch.no_grad():
        x = text_encoder.model(
            **{
                a: b.to(device)
                for a, b in tokenizer.apply_chat_template(
                    [{"role": "user", "content": inp["text"]}],
                    return_tensors="pt",
                    thinking=False,
                    tokenize=True,
                )
            },
            output_hidden_states=True,
        )
    x = torch.cat([x.hidden_states[i] for i in (9, 18, 27)], dim=-1)[0]
    return {"transformer_input": x}

In [ ]:
train_dataset = dataset["train"].map(encode_prompts)

In [ ]:
# Verify encoded shape
torch.tensor(train_dataset[0]["transformer_input"]).shape

In [ ]:
train_dataset = train_dataset.remove_columns(["text"])

## Push to Hub

In [ ]:
train_dataset.push_to_hub("Pankaj121212/alphonse_mucha_encoded_dataset")

In [ ]:
print("✅ Dataset encoded and pushed to hub successfully!")